In [71]:
import json
from pathlib import Path

In [72]:
def split_log_by_json(filepath):
    chunks = []
    current_chunk = []

    with open(filepath, "r") as f:
        for line in f:
            if ".json" in line:
                if current_chunk:
                    chunks.append(current_chunk)
                current_chunk = [line]
            else:
                current_chunk.append(line)

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

In [73]:
def make_json(log_file):
    chunk_lst = split_log_by_json(log_file)
    raw_keywords = ['RETRY', 'FILENAME', 'STRING', 'PORT', 'OM_KEY', 'DEVICE', 'ILLUMINATION', 'LED_ON', 'LED_OFF', 'CAMERA', 'CAPTURE_START', 'CAPTURE_FINISH', 'ERRORS']
    
    result  = [] # Dictionary that will be transformed into .json
    for i, chunk in enumerate(chunk_lst):
        if ".json" in chunk[0]:
            d = {}
            json_fname = chunk[0].split(':')[-1].strip().split('/')[-1]
            json_lst = chunk[5].split('|',1)[1].strip()
            contents = chunk[1:]
            for j, line in enumerate(contents):
                if "TIMESTAMP" in line:
                    for keyword in raw_keywords:
                        if keyword in line:
                            # Scrape log file
                            key = line.removeprefix("[TIMESTAMP]").split(':', 1)[0].strip()
                            value = line.removeprefix("[TIMESTAMP]").split(':', 1)[1].strip()
    
                            if value != 'None':
                                # Scrape data from raw filename
                                if 'FILENAME' in key:
                                    key = 'RAW_FNAME'
                                    value = value.split("/")[-1]
                                    elements = raw_fname.split("_")
                                    i_Run = 0
                                    i_String = 0
                                    i_Exposure = 0
                                    for a, e in enumerate(elements):
                                        if 'exposure' in e: # EXPOSURE
                                            d["EXPOSURE"] = e.removeprefix('exposure').removesuffix('ms')
                                            i_Exposure = a
                                        
                                        if 'gain' in e: d["GAIN"] = e.replace('gain','') # GAIN
                                        
                                        # RUN_TYPE
                                        if 'Camera-Run' in e: i_Run = a
                                        if 'string' in e: i_String = a
                                        d["RUN_TYPE"] = '_'.join(elements[i_Run+1:i_String])
        
                                        # RAW_PATH
                                        date = elements[i_Exposure+1].split('-')[0].removeprefix('2026')
                                        d["RAW_PATH"] = '/data/exp/IceCube/2026/internal-system/upgrade-camera/' + date + '/'
                                    
                                # Clean data in ILLUMINATION and CAMERA keys
                                elif 'ILLUMINATION' in key:
                                    value = value.removeprefix("LED_")
                                elif 'CAMERA' in key:
                                    value = value.removeprefix("CAM_")
                                  
                                d[key] = value
                            else:
                                if 'FILENAME' in key:
                                    d['RAW_FNAME'] = 'None'
                                    d['RAW_PATH'] = 'None'
                                    d['EXPOSURE'] = 'None'
                                    d['GAIN'] = 'None'
                                    d['RUN_TYPE'] = 'None'
                                else:
                                    d[key] = 'None'
                                    
                            d["CONFIG_FNAME"] = json_fname
                            d["CONFIG_LST"] = json_lst
                            result.append(d)

    output_json = log_file.with_name(log_file.stem + "_parsed.json")
    with open(output_json, "w") as f:
        json.dump(result, f, indent=4)          

In [79]:
folder = 'Undated_Logs_JSON' # manually input folder name
for file in Path(folder).glob("*.log"):
    make_json(file)